In [1]:
from slither.slither import Slither
import sys
sys.path.append("../")

from src.detectors import ACM
from src.response.ACM_response import build_mint_access_response

In [2]:
def run_ACM_module(chain: str, address: str) -> dict:

    prefix=''
    if chain=="arbitrum": prefix = 'arbi:'
    if chain=="base": prefix = 'base:'
    if chain=='optimism': prefix = 'optim:'

    try:
        sl = Slither(prefix+address, etherscan_api_key="BK15VKTGEUVYXYXVRQ5H93HWUITNF6ZK8C")
    except Exception as e :
        print(e)
        raise(e)

    ACM_results = ACM.run(sl)

    response_AC = build_mint_access_response(chain, address, ACM_results )
  
    return ACM_results, response_AC

In [3]:
# run on local file : 
! solc-select use 0.8.20
result, response = run_ACM_module(chain='local', address='../test-contracts/mint_20.sol')

print(result)

response

Switched global version to 0.8.20
MyToken.adminMintTokens(uint256)

MyToken.ownerMintTokens(uint256)
❌ High Risk: Unbounded Admin Mint in  MyToken.ownerMintTokens(uint256)

MyToken.mintTokenPayable(uint256)
❌ High Risk: No Access Control Checks
✅ Economic Gate in public Mint in  MyToken.mintTokenPayable(uint256)

MyToken.mintToken(uint256)
❌ High Risk: No Access Control Checks
❌ High Risk: No Economic Gate in public Mint in  MyToken.mintToken(uint256)

{'unbounded_admin_mint': ['MyToken.ownerMintTokens(uint256)'], 'public_mint_without_economic_gate': ['MyToken.mintToken(uint256)'], 'weak_access_control': []}


{'chain': 'local',
 'address': '../test-contracts/mint_20.sol',
 'issues_found': [{'ID': 'ACM-001',
   'Type': 'Access-Control-Mint',
   'Category': 'Unbounded admin mint ',
   'Title': 'ACM-001 — Unbounded admin mint  in `ownerMintTokens(...)`',
   'Severity': 'HIGH',
   'Description': 'MyToken.ownerMintTokens(uint256) allows privileged/admin minting without sufficient governance guardrails (e.g., multi-sig, timelock, or explicit caps). This can enable large or repeated mints if the admin key is compromised or misused.',
   'Function': 'MyToken.ownerMintTokens(uint256)'},
  {'ID': 'ACM-002',
   'Type': 'Access-Control-Mint',
   'Category': 'Public mint without economic gate ',
   'Title': 'ACM-002 — Public mint without economic gate  in `mintToken(...)`',
   'Severity': 'HIGH',
   'Description': 'MyToken.mintToken(uint256) exposes a public mint path without an economic gate/backing deposit. For vault/receipt tokens, ensure previewMint/previewDeposit align and assets are enforced. Un-g